In [ ]:
import json
import time
import numpy as np
from numpy.linalg import norm
from pydantic import BaseModel, Field
from ollama import chat, embeddings
from pathlib import Path
from rank_bm25 import BM25Okapi
from sentence_transformers import CrossEncoder

# ----- GLOBAL VARIABLES -----
rag_mode = "hybrid"   # Options: "hybrid", "cross_encoder", "BM25", "Vector"
enable_llm = True    # True: executes the LLM router | False: tests only the RAG system
model = "llama3:latest"
ratio_queries = 1 # (0.1 = 10% of the dataset)
queries_type = "tool_only" # tool_param_all , tool_only , form_queries
top_k = 3 

# ----- FILE PATHS -----
script_folder = Path().absolute()
tools_list_path = script_folder.parent / 'input' / 'documentation' / 'tools_list_exemples.json'
queries_list_path = script_folder.parent / 'input' / 'tool' / f'{queries_type}.json'
results_path = script_folder / 'output' / f'{rag_mode}_{queries_type}_topk{top_k}.json'

In [450]:
def get_vector(text): 
    response = embeddings(model="mxbai-embed-large", prompt=text)
    return response['embedding']

def format_tool_text(tool):
    """Fonction centralisée pour formater le texte d'un outil."""
    name = tool.get('name', '')
    desc = tool.get('description', str(tool))
    tags = ", ".join(tool.get('tags', []))
    examples = " ".join(tool.get('examples', []))
    return f"{name} : {desc}. Tags {tags}. Examples {examples}."

def precompute_tool_embeddings(tools_list):
    print("Caching vector embeddings...")
    vecs = []
    for tool in tools_list:
        formatted_text = format_tool_text(tool)
        vector = get_vector(formatted_text)
        vecs.append(vector)
    return np.array(vecs)

def precompute_bm25_cache(tools_list):
    print("Caching BM25 index...")
    tokenized_corpus = []
    for tool in tools_list:
        formatted_text = format_tool_text(tool)
        tokens = formatted_text.lower().split()
        tokenized_corpus.append(tokens)
    return BM25Okapi(tokenized_corpus)


In [451]:
def retrieve_tools(user_prompt, tools_list, top_k, mode, tool_matrix=None, bm25_index=None):
    """Routeur de recherche dynamique avec 4 modes explicites."""
    
    if mode == "vector":
        # 1. FULL VECTORIEL (Rapide, calcul mathématique)
        query_vec = np.array(get_vector(user_prompt))
        cosine_scores = np.dot(tool_matrix, query_vec) / (norm(tool_matrix, axis=1) * norm(query_vec))
        top_indices = np.argsort(cosine_scores)[::-1][:top_k]
        
        results = []
        for i in top_indices:
            results.append({"score": float(cosine_scores[i]), "tool": tools_list[i]})
        return results
        
    elif mode == "bm25":
        # 2. FULL BM25 (Ultra rapide, aucun appel LLM)
        tokenized_query = user_prompt.lower().split(" ")
        lexical_scores = bm25_index.get_scores(tokenized_query)
        top_indices = np.argsort(lexical_scores)[::-1][:top_k]
        
        results = []
        for i in top_indices:
            results.append({"score": float(lexical_scores[i]), "tool": tools_list[i]})
        return results

    elif mode == "hybrid":
        # 3. HYBRIDE AVEC RRF (State-of-the-Art)
        # Vecteur
        query_vec = np.array(get_vector(user_prompt))
        vector_scores = np.dot(tool_matrix, query_vec) / (norm(tool_matrix, axis=1) * norm(query_vec))
        # BM25
        tokenized_query = user_prompt.lower().split(" ")
        lexical_scores = bm25_index.get_scores(tokenized_query)

        # RRF Fusion
        k_rrf = 60
        vec_ranks = len(vector_scores) - np.argsort(np.argsort(vector_scores))
        lex_ranks = len(lexical_scores) - np.argsort(np.argsort(lexical_scores))
        
        rrf_vector = 1.0 / (k_rrf + vec_ranks)
        rrf_lexical = 1.0 / (k_rrf + lex_ranks)
        
        final_scores = rrf_vector + rrf_lexical
        top_indices = np.argsort(final_scores)[::-1][:top_k]
        
        results = []
        for i in top_indices:
            results.append({
                "score": float(final_scores[i]), 
                "tool": tools_list[i], 
                "vec_rank": int(vec_ranks[i]),
                "bm25_rank": int(lex_ranks[i])
            })
        return results
        
    elif mode == "cross_encoder":
        # 4. CROSS ENCODER (Lourd mais précis)
        pairs = []
        for tool in tools_list:
            formatted_text = format_tool_text(tool)
            pairs.append([user_prompt, formatted_text])
            
        scores = reranker_model.predict(pairs)
        top_indices = np.argsort(scores)[::-1][:top_k]
        
        results = []
        for i in top_indices:
            results.append({"score": float(scores[i]), "tool": tools_list[i]})
        return results

    else:
        raise ValueError(f"Mode RAG non reconnu: {mode}")

In [452]:
class RouteDecision(BaseModel):
    reasoning: str = Field(description="Step-by-step analysis comparing the user's request against the available tools before making a decision.")
    confidence: float = Field(description="Confidence level from 0.0 to 1.0")
    selected_tool: str = Field(description="The exact name of the tool. Return 'none' if no tool matches.")

def agent_router(user_prompt: str, relevant_tools: list, model_name: str) -> RouteDecision:
    tools_formatted = "\n".join([
        f"{t.get('name', 'Unknown')}: {t.get('description', '')} (Tags: {', '.join(t.get('tags', []))})" 
        for t in relevant_tools
    ])
    
    system_prompt = f"""You are a Router Agent expert in medical and dental imaging (CBCT, IOS, MRI).
    Your role is to analyze the user's request and select the most relevant tool from the FILTERED list below.
    If none of these {len(relevant_tools)} tools fit perfectly, return 'none'.

    === FILTERED TOOLS ===
    {tools_formatted}

    === ROUTING GUIDELINES & EXAMPLES ===
    - Pay close attention to subtle differences. For example, if a user specifically asks for "batch processing" or "multiple scans", prioritize tools designed for batching (e.g., batchdentalseg).
    - If a user asks to "segment" or "split" specific teeth, ensure the tool handles instance segmentation (e.g., amasss_cli).
    - If the request is for registration, check if it's CBCT-to-CBCT, MRI-to-CBCT, or intraoral (IOS) and choose the specific tool accordingly.

    Carefully analyze the user's prompt step-by-step in the 'reasoning' field BEFORE selecting the tool. Output strictly matching the JSON schema.
    """
    
    try:
        response = chat(
            model=model_name,
            messages=[
                {'role': 'system', 'content': system_prompt},
                {'role': 'user', 'content': user_prompt},
            ],
            format=RouteDecision.model_json_schema(),
            options={"temperature": 0},
        )
        return RouteDecision.model_validate_json(response.message.content)
    except Exception as e:
        return RouteDecision(selected_tool="error", confidence=0.0, reasoning=f"Error: {str(e)}")

In [453]:
# ----- LOAD FILES -----
with open(tools_list_path, 'r', encoding='utf-8') as f:
    tools_list = json.load(f)

with open(queries_list_path, 'r', encoding='utf-8') as f:
    queries_list = json.load(f)

# ----- CACHE -----
t_start_setup = time.time()
tool_matrix = None
bm25_index = None

reranker_model = None
if rag_mode.lower() == "cross_encoder":
    print("Loading CrossEncoder model (BAAIbge-reranker-v2-m3)...")
    reranker_model = CrossEncoder("BAAI/bge-reranker-v2-m3", max_length=512)

if rag_mode.lower() in ["vector", "hybrid"]:
    tool_matrix = precompute_tool_embeddings(tools_list)

if rag_mode.lower() in ["bm25", "hybrid"]:
    bm25_index = precompute_bm25_cache(tools_list)

print(f"\nSetup completed in {time.time() - t_start_setup:.2f}s (Mode: {rag_mode.upper()})")

Caching vector embeddings...
Caching BM25 index...

Setup completed in 0.54s (Mode: HYBRID)


In [454]:
def print_rag_block(index, total_queries, prompt, rag_results, mode, expected_tool, rag_match, rag_perfect_match):
    # L'évaluation est déjà faite, on détermine juste l'icône
    if rag_perfect_match:
        rag_hit_sign = "✅"
    elif rag_match:
        rag_hit_sign = "☑️"
    else:
        rag_hit_sign = "❌"

    print(f"\n{'='*65}")
    print(f"Prompt ({index}/{total_queries}) : '{prompt}'")
    print(f"{'-'*65}")
    print(f"I: RAG SEARCH ({mode.upper()}) {rag_hit_sign}")
    print(f"{'-'*65}")
    
    for i, res in enumerate(rag_results, start=1):
        tool_name = res["tool"].get('name', 'Unknown')
        
        # Flèche (->) devant le tool si c'est la bonne réponse
        is_expected = "->" if tool_name == expected_tool else "  "
        
        if mode == "hybrid":
            print(f"{is_expected} {i} Score: {res['score']:.4f} (Vec Rank: {res['vec_rank']}, BM25 Rank: {res['bm25_rank']}) tool: {tool_name}")
        else:
            print(f"{is_expected} {i} Score: {res['score']:.4f} tool: {tool_name}")
    print(f"{'-'*65}")

def print_llm_block(decision, expected_tool, llm_status_icon, latency):
    """Handles the terminal output for the LLM routing decision."""
    print("II: LLM ROUTER DECISION")
    print(f"{'-'*65}")  
    print(f"Tool chosen: {decision.selected_tool} {llm_status_icon} ")
    print(f"Expected   : {expected_tool}")
    print(f"Confidence : {decision.confidence * 100:.2f}%")
    print(f"Latency    : {latency:.2f} seconds")
    print(f"Reasoning  : {decision.reasoning}")

In [455]:
limit = int(len(queries_list) * ratio_queries)
queries_to_run = queries_list[:limit]
total_queries = len(queries_to_run)

# --- Initialize tracking variables ---
rag_correct_count = 0
rag_perfect_count = 0
llm_correct_count = 0

total_rag_latency = 0
total_llm_latency = 0
total_time = 0

results_detail = []

for idx, query_data in enumerate(queries_to_run, 1):
    
    user_prompt = query_data[0]
    expected_tool = query_data[1]
    
    # 1. RAG STEP
    t_start_rag = time.time()
    rag_results = retrieve_tools(user_prompt, tools_list, top_k, rag_mode.lower(), tool_matrix, bm25_index)
    rag_latency = time.time() - t_start_rag
    total_rag_latency += rag_latency
    
    retrieved_tools = []
    for res in rag_results:
        tool_dict = res["tool"]
        tool_name = tool_dict.get("name")
        retrieved_tools.append(tool_name)

    rag_match = expected_tool in retrieved_tools
    rag_perfect_match = (len(retrieved_tools) > 0 and retrieved_tools[0] == expected_tool)
    
    if rag_match: rag_correct_count += 1
    if rag_perfect_match: rag_perfect_count += 1

    print_rag_block(idx, total_queries, user_prompt, rag_results, rag_mode, expected_tool, rag_match, rag_perfect_match)
    
    # 2. LLM STEP (Conditional)
    decision = None
    llm_latency = 0
    llm_match = False
    
    if enable_llm:        
        t0 = time.time()
        decision = agent_router(user_prompt, retrieved_tools, model)
        llm_latency = time.time() - t0
        total_llm_latency += llm_latency
        
        llm_match = (decision.selected_tool == expected_tool)
        if llm_match:
            llm_correct_count += 1
            
        icon = "✅" if llm_match else "❌"
        print_llm_block(decision, expected_tool, icon, llm_latency)
        
    total_query_latency = rag_latency + llm_latency
    total_time += total_query_latency
    
    # Store iteration details
    results_detail.append({
        "query": user_prompt,
        "expected_tool": expected_tool,
        "rag_tools": retrieved_tools,
        "rag_success": rag_match,
        "rag_perfect_success": rag_perfect_match,
        "rag_latency": round(rag_latency, 4),
        "llm_selected_tool": decision.selected_tool if decision else None,
        "llm_success": llm_match,
        "llm_latency": round(llm_latency, 4),
        "total_latency": round(total_query_latency, 4),
        "reasoning": decision.reasoning if decision else None
    })


Prompt (1/843) : 'Locate landmarks on CBCT_patient1'
-----------------------------------------------------------------
I: RAG SEARCH (HYBRID) ✅
-----------------------------------------------------------------
-> 1 Score: 0.0328 (Vec Rank: 1, BM25 Rank: 1) tool: ali_cbct
   2 Score: 0.0320 (Vec Rank: 3, BM25 Rank: 2) tool: ali_ios
   3 Score: 0.0318 (Vec Rank: 2, BM25 Rank: 4) tool: semi_aso_cbct
-----------------------------------------------------------------

Prompt (2/843) : 'mark points A, B, S and N'
-----------------------------------------------------------------
I: RAG SEARCH (HYBRID) ✅
-----------------------------------------------------------------
-> 1 Score: 0.0323 (Vec Rank: 3, BM25 Rank: 1) tool: ali_cbct
   2 Score: 0.0323 (Vec Rank: 2, BM25 Rank: 2) tool: ali_ios
   3 Score: 0.0315 (Vec Rank: 4, BM25 Rank: 3) tool: semi_aso_cbct
-----------------------------------------------------------------

Prompt (3/843) : 'I would like to identify anatomical reference points on

In [456]:
# ==========================================
# FINAL METRICS CALCULATION & EXPORT
# ==========================================
# Calculate accuracies
rag_accuracy = (rag_correct_count / total_queries) * 100 if total_queries > 0 else 0.0
rag_perfect_accuracy = (rag_perfect_count / total_queries) * 100 if total_queries > 0 else 0.0
llm_accuracy = (llm_correct_count / total_queries) * 100 if total_queries > 0 else 0.0

avg_rag_latency = (total_rag_latency / total_queries) if total_queries > 0 else 0.0
avg_llm_latency = (total_llm_latency / total_queries) if total_queries > 0 else 0.0
avg_total_latency = (total_time / total_queries) if total_queries > 0 else 0.0

# Structure the final output JSON report
summary = {
    "rag_mode": rag_mode,
    "enable_llm": enable_llm,
    "queries_type": queries_type,
    "top_k": top_k,
    "model": model,
    "metrics": {
        "total_queries": total_queries,
        "rag_correct": rag_correct_count,
        "rag_accuracy": round(rag_accuracy, 2),
        "rag_perfect_accuracy": round(rag_perfect_accuracy, 2),
        "llm_correct": llm_correct_count,
        "llm_accuracy": round(llm_accuracy, 2),
        "average_rag_latency": round(avg_rag_latency, 4),
        "average_llm_latency": round(avg_llm_latency, 4),
        "average_total_latency": round(avg_total_latency, 4),
        "total_time": round(total_time, 4),
    },
    "details": results_detail,
}

# ----- EXPORT RESULTS -----
results_path.parent.mkdir(parents=True, exist_ok=True)
with open(results_path, 'w', encoding='utf-8') as f:
    json.dump(summary, f, indent=4, ensure_ascii=False)

# ----- FINAL SUMMARY OUTPUT -----
print("\n BENCHMARK COMPLETED ")
print(f"RAG Accuracy (Top {top_k}): {rag_accuracy:.2f}% ({rag_correct_count}/{total_queries})")
print(f"RAG Perfect Accuracy (Top 1): {rag_perfect_accuracy:.2f}% ({rag_perfect_count}/{total_queries})")
if enable_llm:
    print(f"LLM Accuracy (Exact): {llm_accuracy:.2f}% ({llm_correct_count}/{total_queries})")
print(f"Average RAG Latency:  {avg_rag_latency:.4f} sec")
if enable_llm:
    print(f"Average LLM Latency:  {avg_llm_latency:.4f} sec")
    print(f"Average Total Latency:{avg_total_latency:.4f} sec")
print(f"Report saved to:      {results_path.name}")


 BENCHMARK COMPLETED 
RAG Accuracy (Top 3): 86.60% (730/843)
RAG Perfect Accuracy (Top 1): 70.46% (594/843)
Average RAG Latency:  0.0209 sec
Report saved to:      hybrid_form_queries_topk3.json
